# CoDraft — run an LLM

**Change `LLM` and `TASK` in the next cell, then Save & Run All.** Nothing else needs editing.

| TASK | what it does | output |
|---|---|---|
| `enrich` | CoDraft attributes (Nature, Purpose, expanded name) for every term, with the same prompt and schema as the Gemini run | the `codraft_<llm>/` data variant and the attribute file |
| `classify` | the LLM labels each test pair directly, zero-shot and 10-shot, from the name only (`plain`) and from the name + its own attributes (`own`) | `llm_<llm>_<k>shot_<plain/own>` rows for the results table |

`LLM` is a tag from `config/llms.py` (printed below) or any Hugging Face id:

| tag | model |
|---|---|
| `qwen2.5-7b` | Qwen/Qwen2.5-7B-Instruct |
| `qwen3-8b` | Qwen/Qwen3-8B (thinking off) |
| `llama3.1-8b` | meta-llama/Llama-3.1-8B-Instruct (gated) |
| `gemma3-4b` | google/gemma-3-4b-it (gated; fp32, since vLLM refuses Gemma in fp16 and T4 has no bf16) |
| `gemma2-2b` | google/gemma-2-2b-it (gated; fp32; fallback) |
| `nemotron-nano-8b` | nvidia/Llama-3.1-Nemotron-Nano-8B-v1 |
| `gemini-2.5-flash` | the original source, through the API |

**On Kaggle:** accelerator **GPU T4 x2** (the model is split over both cards), Internet on,
the `codraft` dataset attached. Gated models (Llama, Gemma) need their licence accepted on
Hugging Face and a Kaggle secret `HF_TOKEN`; Gemini needs a secret `GEMINI_API_KEY`.

**Run `SMOKE = True` first for every new model** (45 terms, 50 pairs, a few minutes after
loading): it shows whether the model loads on T4 and answers sensibly.

When the session ends, **download one file**: `/kaggle/working/codraft_llm_<llm>_<tasks>.zip`.
For runs 21–29 (`scripts/run.ipynb`), attach this notebook's output as a dataset next to the
data: the runner finds `codraft_<llm>/` under `/kaggle/input`.

In [ ]:
# ===================== the lines to change =====================
LLM = "qwen2.5-7b"
TASK = ["enrich", "classify"]
# ===============================================================

# Optional:
#   SMOKE     45 terms (one per NICE class) and 50 test pairs: checks the model and the
#             pipeline before spending hours. Outputs are prefixed SMOKE_.
#   SCOPE     enrich "all" (7,724 terms: probe + retraining) or "test" (2,928: probe only)
#   VARIANTS  classify inputs: "plain" = the product name only; "own" = name + the Nature /
#             Purpose this LLM made in TASK "enrich" (same session, or an attached run).
#             Add "codraft" to also classify on Gemini's attributes (the trained models' input).
#             Few-shot on "own" needs SCOPE = "all", for demonstrations from train.
#   SHOTS     number of demonstrations, two per label
#   VLLM_PIP  the vLLM to install: 0.30.0 is the version that ran on Kaggle's T4 x2
#             (Qwen2.5-7B smoke test, 2026-10-04)
SMOKE = False
SCOPE = "all"
VARIANTS = ["plain", "own"]
SHOTS = [0, 10]
VLLM_PIP = "vllm==0.30.0"

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/CryAndRRich/CoDraft-Bench.git"
BRANCH = "new"
DATA_ROOT = "/kaggle/input/datasets/nhtquyn/codraft"   # searched for if not found here

def sh(*cmd, **kw):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)

if os.path.isdir("/kaggle/working"):
    OUT_ROOT = "/kaggle/working"
    REPO_DIR = f"{OUT_ROOT}/CoDraft-Bench"
    if os.path.isdir(REPO_DIR):
        sh("git", "-C", REPO_DIR, "pull", "--ff-only")
    else:
        sh("git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR)
    # vLLM brings its own torch / transformers; the training requirements are not installed.
    sh(sys.executable, "-m", "pip", "install", "-q", VLLM_PIP)
    # The image's torchaudio is built for the old torch's CUDA; transformers imports it when
    # present and then fails. No text model needs it.
    sh(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio")
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-llm.txt")
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ("HF_TOKEN", "GEMINI_API_KEY"):
            try:
                os.environ[key] = secrets.get_secret(key)
                print(f"secret {key}: set")
            except Exception:
                print(f"secret {key}: not set")
    except ImportError:
        pass
else:
    # Local: run from anywhere inside the repo, with the splits in data/.
    REPO_DIR = os.getcwd()
    while REPO_DIR != "/" and not os.path.isfile(os.path.join(REPO_DIR, "llm_runner.py")):
        REPO_DIR = os.path.dirname(REPO_DIR)
    OUT_ROOT, DATA_ROOT = REPO_DIR, None

os.chdir(REPO_DIR)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"repo {REPO_DIR} | branch {branch} | commit {commit}")
assert branch == BRANCH, f"expected branch {BRANCH}, got {branch}"

In [ ]:
# In a fresh process, so it imports the torch that vLLM just installed. The model is served
# by vLLM on both GPUs for the length of the run and stopped at the end.
if DATA_ROOT and not os.path.isdir(DATA_ROOT):
    DATA_ROOT = None                      # llm_runner searches /kaggle/input
tasks = [TASK] if isinstance(TASK, str) else list(TASK)
cmd = [sys.executable, "-u", "llm_runner.py", "--llm", LLM, "--task", *tasks,
       "--scope", SCOPE, "--variants", *VARIANTS, "--shots", *map(str, SHOTS),
       "--out-root", OUT_ROOT]
if DATA_ROOT:
    cmd += ["--data-root", DATA_ROOT]
if SMOKE:
    cmd.append("--smoke")
sh(sys.executable, "llm_runner.py", "--list")
sh(*cmd, env={**os.environ, "PYTHONUNBUFFERED": "1"})

## Result

In [ ]:
import glob, json

sys.path.insert(0, REPO_DIR)
from config.llms import safe_tag
prefix = "SMOKE_" if SMOKE else ""
tag = prefix + safe_tag(LLM)

meta = f"{OUT_ROOT}/attributes/{tag}_enrich.json"
if os.path.isfile(meta):
    r = json.load(open(meta))
    print(f"enrich {r['tag']}: {r['n_done']}/{r['n_terms']} terms, {r['n_failed']} failed, "
          f"by mode {r['by_mode']}, generic natures {r['generic_nature']}, "
          f"{r['completion_tokens'] / 1e6:.2f} M output tokens, {r['seconds'] / 60:.1f} min")
    print(f"  revision {r['revision']} | variant splits written: {r['variant_splits']}")

for f in sorted(glob.glob(f"{OUT_ROOT}/results/{prefix}llm_{safe_tag(LLM)}_*_metrics.json")):
    r = json.load(open(f))
    print(f"\n{r['tag']}: Macro-F1 {r['f1_macro']*100:.2f} | QWK {r['qwk']:.4f} | MAE {r['mae']:.4f} | "
          f"n={r['n_test']} | unanswered {r['settings']['n_failed']}")
    print("  per-class F1: " + " / ".join(f"{c['name']} {c['f1']*100:.2f}" for c in r["per_class"]))

zips = sorted(glob.glob(f"{OUT_ROOT}/*codraft_llm_*.zip"), key=os.path.getmtime)
if zips:
    print(f"\nDOWNLOAD THIS: {zips[-1]}  ({os.path.getsize(zips[-1]) / 2**20:.1f} MB)")
else:
    print("\nno zip found; the run cell may have failed before packaging")